In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from pandas import DataFrame

# Current Stock Cleaning

In [ ]:
curr_stocks = pd.read_csv('/content/drive/MyDrive/DATA PROJECT/DS Data Set - Current Stock.csv')
curr_stocks.head()

,Product ID,Brand,Product Name,Current Stock Qty,Unit Cost (INR),Last Restock Date
0,P0224,BRAND_C,145/70 R 12 BY-801,15,"2,766.00",1/20/2026
1,P0222,BRAND_C,145/80 R 12 BY-801,364,"2,876.00",1/20/2026
2,P0258,BRAND_C,155/65 R 13 BY-801,191,"2,976.00",1/20/2026
3,P0228,BRAND_C,145/80 R 13 BY-801,26,"3,067.00",12/22/2025
4,P0168,BRAND_A,145/80 R 13 TR203,167,"3,245.00",6/16/2025


In [ ]:
curr_stocks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253 entries, 0 to 252
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Product ID         253 non-null    object
 1   Brand              253 non-null    object
 2   Product Name       253 non-null    object
 3   Current Stock Qty  253 non-null    int64 
 4   Unit Cost (INR)    251 non-null    object
 5   Last Restock Date  253 non-null    object
dtypes: int64(1), object(5)
memory usage: 12.0+ KB


In [ ]:
curr_stocks['Product ID'].nunique()

253

In [ ]:
curr_stocks = curr_stocks.rename(columns= {'Unit Cost (INR)':'Unit Cost'})
curr_stocks.head()

,Product ID,Brand,Product Name,Current Stock Qty,Unit Cost,Last Restock Date
0,P0224,BRAND_C,145/70 R 12 BY-801,15,"2,766.00",1/20/2026
1,P0222,BRAND_C,145/80 R 12 BY-801,364,"2,876.00",1/20/2026
2,P0258,BRAND_C,155/65 R 13 BY-801,191,"2,976.00",1/20/2026
3,P0228,BRAND_C,145/80 R 13 BY-801,26,"3,067.00",12/22/2025
4,P0168,BRAND_A,145/80 R 13 TR203,167,"3,245.00",6/16/2025


1. Handling date format inconsistencies

In [ ]:
#Check delimiters
date_delimiters = curr_stocks['Last Restock Date'].str.replace(r'[a-zA-Z0-9]', '', regex=True).value_counts()
print(date_delimiters)

Last Restock Date
//    188
--     65
Name: count, dtype: int64


In [ ]:
#Checks varying length
date_len = curr_stocks['Last Restock Date'].str.len().value_counts()
date_len

,count
Last Restock Date,
9,131
8,65
10,57


In [ ]:
#Group dates with varying size and formats
for length, group in curr_stocks.groupby(curr_stocks['Last Restock Date'].str.len()):
    print(f"\n--- Length: {length} (Total rows: {len(group)}) ---")

    # Check which delimiter this length uses
    has_slash = group['Last Restock Date'].str.contains('/').any()
    has_dash = group['Last Restock Date'].str.contains('-').any()

    print(f"Contains Slashes: {has_slash} | Contains Dashes: {has_dash}")
    print("Sample rows:")
    print(group['Last Restock Date'].head(3).to_string(index=False))


--- Length: 8 (Total rows: 65) ---
Contains Slashes: False | Contains Dashes: True
Sample rows:
11-06-25
02-11-23
01-06-26

--- Length: 9 (Total rows: 131) ---
Contains Slashes: True | Contains Dashes: False
Sample rows:
1/20/2026
1/20/2026
1/20/2026

--- Length: 10 (Total rows: 57) ---
Contains Slashes: True | Contains Dashes: False
Sample rows:
12/22/2025
12/22/2025
12/22/2025


In [ ]:
# Standardizing date format
# Step 1: Handle the slash formats (Lengths 9 and 10)
# %m handles both 1-digit and 2-digit months seamlessly
slash_parsed = pd.to_datetime(
    curr_stocks['Last Restock Date'], format="%m/%d/%Y", errors="coerce"
)

# Step 2: Handle the short dash format (Length 8)
# %y (lowercase) correctly reads 2-digit years like '25' as 2025
dash_parsed = pd.to_datetime(
    curr_stocks['Last Restock Date'], format="%m-%d-%y", errors="coerce"
)

# Step 3: Combine them together (if slash fails, use dash)
curr_stocks["final_clean_date"] = slash_parsed.fillna(dash_parsed)

print(curr_stocks)

    Product ID    Brand                   Product Name  Current Stock Qty  \
0        P0224  BRAND_C             145/70 R 12 BY-801                 15   
1        P0222  BRAND_C             145/80 R 12 BY-801                364   
2        P0258  BRAND_C             155/65 R 13 BY-801                191   
3        P0228  BRAND_C             145/80 R 13 BY-801                 26   
4        P0168  BRAND_A              145/80 R 13 TR203                167   
..         ...      ...                            ...                ...   
248      P0001  BRAND_D     1000 R 20/18 PR CT23 (SET)                719   
249      P0013  BRAND_D    315/80 R 22.5 /20 PR (CD01)                 84   
250      P0011  BRAND_D    315/80 R 22.5 /20 PR (CT23)                 24   
251      P0166  BRAND_A               145/80 R 12 TR17                 76   
252      P0119  BRAND_A  265/75 R 16/ 10PR LT TR29 OWL                 12   

     Unit Cost Last Restock Date final_clean_date  
0     2,766.00         

In [ ]:
curr_stocks = curr_stocks.drop(columns=['Last Restock Date'])
curr_stocks.head()

,Product ID,Brand,Product Name,Current Stock Qty,Unit Cost,final_clean_date
0,P0224,BRAND_C,145/70 R 12 BY-801,15,"2,766.00",2026-01-20
1,P0222,BRAND_C,145/80 R 12 BY-801,364,"2,876.00",2026-01-20
2,P0258,BRAND_C,155/65 R 13 BY-801,191,"2,976.00",2026-01-20
3,P0228,BRAND_C,145/80 R 13 BY-801,26,"3,067.00",2025-12-22
4,P0168,BRAND_A,145/80 R 13 TR203,167,"3,245.00",2025-06-16


In [ ]:
print(curr_stocks['final_clean_date'].max())
print(curr_stocks['final_clean_date'].min())

2026-02-11 00:00:00
2019-12-24 00:00:00


In [ ]:
curr_stocks = curr_stocks.rename(columns={'final_clean_date':'Last Restock Date'})
curr_stocks.head()

,Product ID,Brand,Product Name,Current Stock Qty,Unit Cost,Last Restock Date
0,P0224,BRAND_C,145/70 R 12 BY-801,15,"2,766.00",2026-01-20
1,P0222,BRAND_C,145/80 R 12 BY-801,364,"2,876.00",2026-01-20
2,P0258,BRAND_C,155/65 R 13 BY-801,191,"2,976.00",2026-01-20
3,P0228,BRAND_C,145/80 R 13 BY-801,26,"3,067.00",2025-12-22
4,P0168,BRAND_A,145/80 R 13 TR203,167,"3,245.00",2025-06-16


2. Changing column data type from object to numerical

In [ ]:
# Cast to string to safely use string methods
unit_cost_str = curr_stocks['Unit Cost'].astype(str)

In [ ]:
unit_cost_str = unit_cost_str.str.replace(',', '', regex=False)
unit_cost_str = unit_cost_str.str.strip()

In [ ]:
# Changing data type to numerical
curr_stocks['Unit Cost'] = pd.to_numeric(unit_cost_str, errors='coerce')

In [ ]:
curr_stocks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253 entries, 0 to 252
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Product ID         253 non-null    object        
 1   Brand              253 non-null    object        
 2   Product Name       253 non-null    object        
 3   Current Stock Qty  253 non-null    int64         
 4   Unit Cost          251 non-null    float64       
 5   Last Restock Date  253 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 12.0+ KB


In [ ]:
curr_stocks.to_csv('current_stocks.csv', index=False)

# Sales Table Cleaning

In [ ]:
sales = pd.read_csv('/content/drive/MyDrive/DATA PROJECT/DS Data Set - Sales.csv')
sales.head()

,Date,Brand,Product ID,Product Name,Units Sold,Unit Price (INR),Revenue (INR),Transaction Type,Price Level
0,08-04-24,BRAND_A,P0102,215/70 R 16 TR28 OWL,4,"15,124.00","60,496.00",Sale,A
1,08-04-24,BRAND_A,P0144,33 x 12.5 R 15/ 6PR OWL TR28,1,"19,658.00","19,658.00",Sale,X
2,08-04-24,BRAND_A,P0207,145 R 12C / 8PR TR06,2,"7,068.00","14,136.00",Sale,A
3,08-04-24,BRAND_B,P0378,195/55 R 16 HB-2,2,"5,158.00","10,315.00",Sale,A
4,08-04-24,BRAND_B,P0382,225/55 R 16 HB-2,4,"16,209.00","64,836.00",Sale,A


In [ ]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16999 entries, 0 to 16998
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Date              16999 non-null  object
 1   Brand             16999 non-null  object
 2   Product ID        16999 non-null  object
 3   Product Name      16999 non-null  object
 4   Units Sold        16999 non-null  int64 
 5   Unit Price (INR)  16999 non-null  object
 6   Revenue (INR)     16999 non-null  object
 7   Transaction Type  16999 non-null  object
 8   Price Level       16999 non-null  object
dtypes: int64(1), object(8)
memory usage: 1.2+ MB


1. Handiling numerical columns

In [ ]:
# Changing column names
sales = sales.rename(columns={'Revenue (INR)':'Revenue','Unit Price (INR)':'Unit Price'})

In [ ]:
# Cast to string to safely use string methods
revenue_str = sales['Revenue'].astype(str)
unit_price_str = sales['Unit Price'].astype(str)

In [ ]:
# 2. Remove commas, currency symbols, and extra spaces
revenue_str = revenue_str.str.replace(',', '', regex=False)
revenue_str = revenue_str.str.strip()

unit_price_str = unit_price_str.str.replace(',', '', regex=False)
unit_price_str = unit_price_str.str.strip()

In [ ]:
# Changing to numeric column
sales['Revenue'] = pd.to_numeric(revenue_str, errors='coerce')
sales

,Date,Brand,Product ID,Product Name,Units Sold,Unit Price,Revenue,Transaction Type,Price Level
0,08-04-24,BRAND_A,P0102,215/70 R 16 TR28 OWL,4,"15,124.00",60496.0,Sale,A
1,08-04-24,BRAND_A,P0144,33 x 12.5 R 15/ 6PR OWL TR28,1,"19,658.00",19658.0,Sale,X
2,08-04-24,BRAND_A,P0207,145 R 12C / 8PR TR06,2,"7,068.00",14136.0,Sale,A
3,08-04-24,BRAND_B,P0378,195/55 R 16 HB-2,2,"5,158.00",10315.0,Sale,A
4,08-04-24,BRAND_B,P0382,225/55 R 16 HB-2,4,"16,209.00",64836.0,Sale,A
...,...,...,...,...,...,...,...,...,...
16994,09-12-25,BRAND_C,P0220,295/80 R 22.5 /18 PR (VS788),-2,"39,400.00",-78800.0,Return,A
16995,23-02-26,BRAND_C,P0221,295/80 R 22.5 /18 PR (VS806),-4,"40,200.00",-160800.0,Return,A
16996,28-02-26,BRAND_C,P0221,295/80 R 22.5 /18 PR (VS806),-4,"40,200.00",-160800.0,Return,A
16997,18-12-25,BRAND_C,P0326,825 R 16/16 PR LO301 (SET),-2,"29,160.00",-58320.0,Return,A


In [ ]:
sales['Unit Price'] = pd.to_numeric(unit_price_str, errors='coerce')
sales

,Date,Brand,Product ID,Product Name,Units Sold,Unit Price,Revenue,Transaction Type,Price Level
0,08-04-24,BRAND_A,P0102,215/70 R 16 TR28 OWL,4,15124.0,60496.0,Sale,A
1,08-04-24,BRAND_A,P0144,33 x 12.5 R 15/ 6PR OWL TR28,1,19658.0,19658.0,Sale,X
2,08-04-24,BRAND_A,P0207,145 R 12C / 8PR TR06,2,7068.0,14136.0,Sale,A
3,08-04-24,BRAND_B,P0378,195/55 R 16 HB-2,2,5158.0,10315.0,Sale,A
4,08-04-24,BRAND_B,P0382,225/55 R 16 HB-2,4,16209.0,64836.0,Sale,A
...,...,...,...,...,...,...,...,...,...
16994,09-12-25,BRAND_C,P0220,295/80 R 22.5 /18 PR (VS788),-2,39400.0,-78800.0,Return,A
16995,23-02-26,BRAND_C,P0221,295/80 R 22.5 /18 PR (VS806),-4,40200.0,-160800.0,Return,A
16996,28-02-26,BRAND_C,P0221,295/80 R 22.5 /18 PR (VS806),-4,40200.0,-160800.0,Return,A
16997,18-12-25,BRAND_C,P0326,825 R 16/16 PR LO301 (SET),-2,29160.0,-58320.0,Return,A


2. Handiling Date column

In [ ]:
sales.head()

,Date,Brand,Product ID,Product Name,Units Sold,Unit Price,Revenue,Transaction Type,Price Level
0,08-04-24,BRAND_A,P0102,215/70 R 16 TR28 OWL,4,15124.0,60496.0,Sale,A
1,08-04-24,BRAND_A,P0144,33 x 12.5 R 15/ 6PR OWL TR28,1,19658.0,19658.0,Sale,X
2,08-04-24,BRAND_A,P0207,145 R 12C / 8PR TR06,2,7068.0,14136.0,Sale,A
3,08-04-24,BRAND_B,P0378,195/55 R 16 HB-2,2,5158.0,10315.0,Sale,A
4,08-04-24,BRAND_B,P0382,225/55 R 16 HB-2,4,16209.0,64836.0,Sale,A


In [ ]:
#Check delimiters
date_delimiters = sales['Date'].str.replace(r'[a-zA-Z0-9]', '', regex=True).value_counts()
print(date_delimiters)

Date
--    16999
Name: count, dtype: int64


In [ ]:
# Standardizing date format
# Handle the short dash format (Length 8)
# %y (lowercase) correctly reads 2-digit years like '25' as 2025
sales['Date'] = pd.to_datetime(
    sales['Date'], format="%d-%m-%y", errors="coerce"
)

In [ ]:
print(sales['Date'].min())
print(sales['Date'].max())

2024-04-08 00:00:00
2026-03-31 00:00:00


In [ ]:
sales.to_csv('sales.csv', index=False)

# **Supplier Table Analysis**

In [ ]:
supplier = pd.read_csv('/content/drive/MyDrive/DATA PROJECT/DS Data Set - Supplier.csv')
supplier.head()

,Brand,Supplier ID,Supplier Name,Avg Lead Time (Days),Lead Time Std Dev (Days),Min Lead Time (Days),Max Lead Time (Days)
0,BRAND_A,SUP0001,Supplier 1,40,6.7,30,70
1,BRAND_D,SUP0002,Supplier 2,60,9.2,45,100
2,BRAND_C,SUP0003,Supplier 3,50,10.0,30,90
3,BRAND_B,SUP0004,Supplier 4,60,11.7,50,120


In [ ]:
supplier.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Brand                     4 non-null      object 
 1   Supplier ID               4 non-null      object 
 2   Supplier Name             4 non-null      object 
 3   Avg Lead Time (Days)      4 non-null      int64  
 4   Lead Time Std Dev (Days)  4 non-null      float64
 5   Min Lead Time (Days)      4 non-null      int64  
 6   Max Lead Time (Days)      4 non-null      int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 356.0+ bytes


In [ ]:
supplier.to_csv('supplier.csv', index=False)

In [ ]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16999 entries, 0 to 16998
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              16999 non-null  datetime64[ns]
 1   Brand             16999 non-null  object        
 2   Product ID        16999 non-null  object        
 3   Product Name      16999 non-null  object        
 4   Units Sold        16999 non-null  int64         
 5   Unit Price        16999 non-null  float64       
 6   Revenue           16999 non-null  float64       
 7   Transaction Type  16999 non-null  object        
 8   Price Level       16999 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 1.2+ MB


In [ ]:
curr_stocks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253 entries, 0 to 252
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Product ID         253 non-null    object        
 1   Brand              253 non-null    object        
 2   Product Name       253 non-null    object        
 3   Current Stock Qty  253 non-null    int64         
 4   Unit Cost          251 non-null    float64       
 5   Last Restock Date  253 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 12.0+ KB
